# Deep Equilibrium Networks
## A Neural Network at Infinite Depth

**Companion notebook** — *Complexity of Deep Computations via Topology* [Dueñez, Iovino, Matos-Wiederhold, Salvetti, Tall]

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/eduenez/public-site/blob/main/2026-FES_Acatlan-ultracomputos/notebooks/deq_networks.ipynb)

---

The standard picture of a deep network is a sequence of layers:
$$z_1 = f_\theta(x), \quad z_2 = f_\theta(z_1), \quad \ldots, \quad z_N = f_\theta(z_{N-1}).$$
Output = $z_N$.  More depth → more expressive power → more parameters.

**A different question.** What if we ran those layers *forever*?
Is there a limit $z_\infty = \lim_{n\to\infty} z_n$, and can we use it directly?

If $z_\infty$ exists, it must satisfy the **fixed-point equation**
$$z^\star = f_\theta(z^\star, x),$$
where $x$ is the external input.  A network whose output is defined as
the solution to this equation — rather than as a finite composition — is a
**Deep Equilibrium Model** (DEQ; Bai, Kolter & Koltun, 2019).

In the language of the research paper, $z^\star$ is a **deep equilibrium**:
the computation state at infinite depth.
This notebook explores when such equilibria exist, when they are unique,
and what "breaks" when they do not.


## The 1-D model: $z^\star = \tanh(w z^\star + x)$

To build geometric intuition, we study the simplest possible case:
- **State**: $z \in \mathbb{R}$ (scalar hidden state)
- **Input**: $x \in \mathbb{R}$ (plays the role of a bias from the input layer)
- **Weight**: $w \in \mathbb{R}$ (a single recurrent weight)
- **Transition**: $f(z) = \tanh(wz + x)$

The deep-equilibrium equation becomes
$$z^\star = \tanh(w z^\star + x).$$

**Graphically**, $z^\star$ is a point where the curve $y = \tanh(wz + x)$
crosses the diagonal $y = z$.
**Iteratively**, one finds $z^\star$ by repeating $z \leftarrow \tanh(wz + x)$
from any starting point — if the sequence converges.

The **cobweb diagram** below makes both interpretations visual.


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
import matplotlib.gridspec as gridspec
import ipywidgets as widgets
from scipy.optimize import brentq
%matplotlib inline


In [ ]:
# ── 1-D DEQ transition ────────────────────────────────────────────────────────

def f(z, w, x):
    return np.tanh(w * z + x)

def df(z, w, x):
    # derivative of f w.r.t. z
    t = np.tanh(w * z + x)
    return w * (1.0 - t**2)


# ── Fixed-point finder ────────────────────────────────────────────────────────

def find_fixed_points(w, x, n_scan=4000, tol=1e-10):
    # Scan [-2, 2] for sign changes of g(z) = tanh(wz+x) - z, then refine.
    zs = np.linspace(-2.0, 2.0, n_scan)
    g  = np.tanh(w * zs + x) - zs
    fps = []
    for i in range(len(zs) - 1):
        if g[i] * g[i + 1] < 0:
            try:
                z_star = brentq(lambda z: np.tanh(w * z + x) - z,
                                zs[i], zs[i + 1], xtol=tol)
                fps.append(z_star)
            except ValueError:
                pass
    return fps


# ── Orbit generator ───────────────────────────────────────────────────────────

def orbit(w, x, z0, n_steps):
    # Return the sequence z_0, z_1, ..., z_{n_steps}.
    zs = [z0]
    z  = z0
    for _ in range(n_steps):
        z = f(z, w, x)
        zs.append(z)
    return np.array(zs)


# ── Cobweb data ───────────────────────────────────────────────────────────────

def cobweb_data(w, x, z0, n_steps):
    # Returns (xs, ys) for the cobweb path.
    # Path: (z0,z0) -> (z0, f(z0)) -> (f(z0),f(z0)) -> (f(z0), f(f(z0))) -> ...
    xs, ys = [z0], [z0]
    z = z0
    for _ in range(n_steps):
        fz = f(z, w, x)
        xs += [z,  fz]
        ys += [fz, fz]
        z   = fz
    return np.array(xs), np.array(ys)


# ── Convergence check ─────────────────────────────────────────────────────────

def convergence_info(w, x, z0=0.0, max_iter=500, tol=1e-8):
    # Run iteration; return (converged, n_iter, z_final).
    z = z0
    for n in range(max_iter):
        z_new = f(z, w, x)
        if abs(z_new - z) < tol:
            return True, n + 1, z_new
        if abs(z_new) > 10.0:
            return False, n + 1, z_new
        z = z_new
    return False, max_iter, z


## Cobweb diagrams: reading the geometry of iteration

A *cobweb diagram* traces the orbit of the iteration $z \mapsto f(z)$:

1. Start at $(z_0, z_0)$ on the diagonal $y = z$.
2. Move **vertically** to $(z_0, f(z_0))$ — landing on the curve.
3. Move **horizontally** to $(f(z_0), f(z_0))$ — landing back on the diagonal.
4. Repeat from the new point $z_1 = f(z_0)$.

The path spirals toward a fixed point when the iteration **converges** (tame)
and spirals outward or oscillates when it does not (wild).

**Use the sliders** to explore how the weight $w$ and input $x$ affect convergence.
Pay attention to what happens near $|w| = 1$: this is the critical threshold.


In [ ]:
@widgets.interact(
    w=widgets.FloatSlider(
        value=0.8, min=-3.0, max=3.0, step=0.05,
        description='Weight w',
        style={'description_width': 'initial'},
        layout=widgets.Layout(width='450px'),
    ),
    x=widgets.FloatSlider(
        value=0.5, min=-3.0, max=3.0, step=0.05,
        description='Input x',
        style={'description_width': 'initial'},
        layout=widgets.Layout(width='450px'),
    ),
    z0=widgets.FloatSlider(
        value=-1.5, min=-2.0, max=2.0, step=0.1,
        description='Start z0',
        style={'description_width': 'initial'},
        layout=widgets.Layout(width='450px'),
    ),
    n_steps=widgets.IntSlider(
        value=12, min=1, max=60, step=1,
        description='Cobweb steps',
        style={'description_width': 'initial'},
        layout=widgets.Layout(width='450px'),
    ),
)
def cobweb_explorer(w, x, z0, n_steps):
    fps = find_fixed_points(w, x)
    cx_xs, cx_ys = cobweb_data(w, x, z0, n_steps)

    zplot = np.linspace(-2.2, 2.2, 600)
    fplot = np.tanh(w * zplot + x)

    fig, axes = plt.subplots(1, 2, figsize=(12, 5))

    # ── Left: cobweb diagram ──────────────────────────────────────────────────
    ax = axes[0]
    ax.plot(zplot, fplot, color='steelblue', lw=2,
            label=r'$y = \tanh(wz + x)$')
    ax.plot(zplot, zplot, color='gray', lw=1.2, ls='--', label=r'$y = z$')
    ax.plot(cx_xs, cx_ys, color='darkorange', lw=1.5, alpha=0.85, label='Cobweb')
    ax.plot(z0, z0, 'o', color='darkorange', ms=7, zorder=5)

    stable_sym   = dict(marker='*', ms=14, zorder=6, markeredgecolor='k',
                        markeredgewidth=0.7)
    unstable_sym = dict(marker='D', ms=9,  zorder=6, markeredgecolor='k',
                        markeredgewidth=0.7, mfc='white')
    for zs in fps:
        deriv = df(zs, w, x)
        stable = abs(deriv) < 1.0
        label = (f'FP: z*={zs:.3f},  |f\' |={abs(deriv):.3f} '
                 + ('(stable)' if stable else '(UNSTABLE)'))
        sym = stable_sym if stable else unstable_sym
        ax.plot(zs, zs, color='limegreen' if stable else 'red',
                label=label, **sym)

    ax.set_xlim(-2.2, 2.2)
    ax.set_ylim(-2.0, 2.0)
    ax.set_xlabel('z', fontsize=12)
    ax.set_ylabel('f(z)  /  z', fontsize=12)
    ax.set_title(f'Cobweb diagram  (w={w:.2f}, x={x:.2f})', fontsize=12)
    ax.legend(fontsize=9, loc='upper left')
    ax.axhline(0, color='k', lw=0.4)
    ax.axvline(0, color='k', lw=0.4)
    ax.grid(True, alpha=0.25)

    # ── Right: orbit / convergence plot ───────────────────────────────────────
    ax2 = axes[1]
    orb = orbit(w, x, z0, n_steps)
    converged, n_conv, z_lim = convergence_info(w, x, z0=z0)

    ax2.plot(range(len(orb)), orb, 'o-', color='darkorange',
             ms=5, lw=1.5, label='Orbit $z_n$')
    if converged and fps:
        nearest = min(fps, key=lambda zs: abs(zs - z_lim))
        ax2.axhline(nearest, color='limegreen', ls='--', lw=1.5,
                    label=f'Fixed point z*={nearest:.4f}')

    ax2.set_xlabel('Depth $n$', fontsize=12)
    ax2.set_ylabel('$z_n$', fontsize=12)
    ax2.set_title(
        f'Orbit  ({"converged in " + str(n_conv) + " steps" if converged else "did NOT converge"})',
        fontsize=12,
    )
    ax2.legend(fontsize=9)
    ax2.grid(True, alpha=0.25)

    plt.tight_layout()
    plt.show()


## When does iteration converge? The contraction condition

**Banach Fixed-Point Theorem.** If $f : \mathbb{R} \to \mathbb{R}$ is a
*contraction* — meaning $|f'(z)| \leq L < 1$ for all $z$ —
then $f$ has a **unique** fixed point $z^\star$, and iteration converges to it
from any starting point, with exponential rate $L$.

For our map $f(z) = \tanh(wz + x)$:
$$f'(z) = w \cdot \operatorname{sech}^2(wz + x) = w(1 - \tanh^2(wz + x)).$$
At a fixed point $z^\star = \tanh(wz^\star + x)$, this simplifies to
$$f'(z^\star) = w(1 - (z^\star)^2).$$

Since $|z^\star| < 1$, we have $1 - (z^\star)^2 \in (0, 1]$, so:
- $|w| < 1$ $\Rightarrow$ $|f'(z^\star)| < 1$: **always a contraction** — unique fixed point, guaranteed convergence.
- $|w| > 1$: the fixed point may be **unstable** ($|f'(z^\star)| > 1$), and iteration may diverge or oscillate.

**Phase diagram below:** the (w, x)-plane is colored by convergence behavior,
revealing a Julia-set-like boundary between the tame and wild regimes.


In [ ]:
# Compute convergence phase diagram in (w, x) space.
# This may take ~10 s at res=180; increase for publication quality.

RES    = 180
W_VALS = np.linspace(-3.0, 3.0, RES)
X_VALS = np.linspace(-3.0, 3.0, RES)
WW, XX = np.meshgrid(W_VALS, X_VALS)

n_iter_grid = np.full((RES, RES), 500, dtype=float)
status_grid  = np.zeros((RES, RES), dtype=int)   # 0=not converged, 1=converged

MAX_ITER = 500
TOL      = 1e-8

Z = np.zeros((RES, RES))   # iterate from z0 = 0

for n in range(MAX_ITER):
    active = status_grid == 0
    if not active.any():
        break
    Z_new = np.tanh(WW * Z + XX)
    diff  = np.abs(Z_new - Z)
    converged_now = active & (diff < TOL)
    status_grid[converged_now]  = 1
    n_iter_grid[converged_now]  = n + 1
    diverged_now = active & (np.abs(Z_new) > 10.0)
    status_grid[diverged_now]   = -1
    n_iter_grid[diverged_now]   = n + 1
    Z = Z_new

# ── Plot ──────────────────────────────────────────────────────────────────────

fig, axes = plt.subplots(1, 2, figsize=(13, 5.5))

# Left: convergence speed (log iterations)
ax = axes[0]
img = n_iter_grid.copy()
img[status_grid != 1] = np.nan     # mask non-converged

im = ax.imshow(
    np.log1p(img), origin='lower',
    extent=[-3, 3, -3, 3],
    cmap='viridis', aspect='equal',
    vmin=0, vmax=np.log1p(MAX_ITER),
)
ax.contour(WW, XX, status_grid, levels=[0.5], colors='white', linewidths=0.8)
ax.axvline(-1, color='red', lw=1, ls='--', alpha=0.7, label='|w|=1')
ax.axvline( 1, color='red', lw=1, ls='--', alpha=0.7)
cbar = plt.colorbar(im, ax=ax)
cbar.set_label('log(1 + iterations to converge)', fontsize=10)
ax.set_xlabel('Weight  w', fontsize=12)
ax.set_ylabel('Input   x', fontsize=12)
ax.set_title('Convergence speed\n(dark = slow, white boundary = non-converged)', fontsize=11)
ax.legend(fontsize=9)

# Right: tame / wild classification
ax2 = axes[1]
color_img = np.zeros((RES, RES, 3))
color_img[status_grid ==  1] = [0.2, 0.7, 0.3]   # green  = converged (tame / NIP)
color_img[status_grid ==  0] = [0.9, 0.7, 0.1]   # yellow = slow / marginal
color_img[status_grid == -1] = [0.85, 0.2, 0.2]  # red    = diverged  (wild / IP)

ax2.imshow(color_img, origin='lower', extent=[-3, 3, -3, 3], aspect='equal')
ax2.axvline(-1, color='white', lw=1.2, ls='--', alpha=0.8, label='|w|=1')
ax2.axvline( 1, color='white', lw=1.2, ls='--', alpha=0.8)
from matplotlib.patches import Patch
legend_elems = [
    Patch(fc=[0.2,0.7,0.3], label='Converged  (tame / NIP)'),
    Patch(fc=[0.9,0.7,0.1], label='Slow / marginal'),
    Patch(fc=[0.85,0.2,0.2], label='Diverged  (wild / IP)'),
]
ax2.legend(handles=legend_elems, fontsize=9, loc='upper right')
ax2.set_xlabel('Weight  w', fontsize=12)
ax2.set_ylabel('Input   x', fontsize=12)
ax2.set_title('Tame vs. wild: convergence phase diagram\n'
              'Compare to Newton fractal: same structure, 1-D version', fontsize=11)

plt.tight_layout()
plt.show()


## From 1-D to Deep Equilibrium Networks

The 1-D picture extends directly to higher dimensions.
In a DEQ layer with hidden state $z \in \mathbb{R}^d$ and input $x \in \mathbb{R}^n$:
$$z^\star = f_\theta(z^\star, x), \qquad
  f_\theta(z, x) = \sigma(Wz + Ux + b),$$
where $W \in \mathbb{R}^{d \times d}$ is the recurrent weight matrix,
$U \in \mathbb{R}^{d \times n}$ is the input weight, and $\sigma$ is a pointwise nonlinearity.

**Forward pass**: solve the fixed-point equation by iteration (or a faster solver).

**Backward pass (implicit differentiation).**
Differentiating $z^\star = f_\theta(z^\star, x)$ implicitly with respect to $\theta$:
$$\frac{\partial z^\star}{\partial \theta}
  = J_f(z^\star) \frac{\partial z^\star}{\partial \theta}
  + \frac{\partial f_\theta}{\partial \theta}\bigg|_{z^\star},$$
where $J_f = \partial f / \partial z |_{z^\star}$ is the Jacobian at the fixed point.
Solving:
$$\frac{\partial z^\star}{\partial \theta}
  = \bigl(I - J_f(z^\star)\bigr)^{-1}
    \frac{\partial f_\theta}{\partial \theta}\bigg|_{z^\star}.$$

**Key insight**: the gradient depends only on $z^\star$ and $J_f(z^\star)$ —
not on the intermediate iterates $z_1, z_2, \ldots$
An explicit $N$-layer network requires $O(N)$ memory to backpropagate;
a DEQ requires $O(1)$, regardless of how many iterations were used.

This is only valid when $I - J_f(z^\star)$ is invertible,
which is guaranteed when the fixed point is **stable**:
$\rho(J_f(z^\star)) < 1$, the Banach contraction condition.


In [ ]:
# Side-by-side: explicit N-layer network vs. DEQ fixed-point iteration
# Both compute the same function; the difference is in memory and backprop.

@widgets.interact(
    w=widgets.FloatSlider(
        value=0.75, min=-0.99, max=0.99, step=0.05,
        description='Weight w',
        style={'description_width': 'initial'},
        layout=widgets.Layout(width='420px'),
    ),
    x=widgets.FloatSlider(
        value=1.0, min=-3.0, max=3.0, step=0.1,
        description='Input x',
        style={'description_width': 'initial'},
        layout=widgets.Layout(width='420px'),
    ),
    max_depth=widgets.IntSlider(
        value=40, min=5, max=200, step=5,
        description='Max depth N',
        style={'description_width': 'initial'},
        layout=widgets.Layout(width='420px'),
    ),
)
def deq_vs_explicit(w, x, max_depth):
    # Explicit: z_n = f(z_{n-1}), starting from z0 = 0
    orb   = orbit(w, x, z0=0.0, n_steps=max_depth)
    z_inf = orb[-1]   # best approximation to z* from explicit stacking

    # DEQ: run iteration to convergence (same sequence, but we only keep z*)
    converged, n_conv, z_star = convergence_info(w, x, z0=0.0,
                                                  max_iter=10_000, tol=1e-12)

    # Implicit gradient  dz*/dw  (from implicit differentiation)
    # Differentiating z* = tanh(w z* + x) w.r.t. w gives:
    #   dz*/dw = sech^2(wz*+x) * z* / (1 - f'(z*))
    # Since z* = tanh(wz*+x), we have sech^2(wz*+x) = 1 - z*^2.
    fp_deriv  = df(z_star, w, x)          # f'(z*) = w(1 - z*^2)
    sech_sq   = 1.0 - z_star**2           # sech^2(wz*+x)
    if abs(1.0 - fp_deriv) > 1e-10:
        grad_implicit = sech_sq * z_star / (1.0 - fp_deriv)
    else:
        grad_implicit = float('inf')

    # Finite-difference gradient for verification
    delta = 1e-5
    _, _, z_plus  = convergence_info(w + delta, x, z0=0.0, tol=1e-12)
    _, _, z_minus = convergence_info(w - delta, x, z0=0.0, tol=1e-12)
    grad_fd = (z_plus - z_minus) / (2 * delta)

    # ── Plot ──────────────────────────────────────────────────────────────────
    fig, axes = plt.subplots(1, 2, figsize=(13, 5))

    # Left: convergence trajectory
    ax = axes[0]
    errors = np.abs(orb - z_star)
    errors = np.where(errors < 1e-15, 1e-15, errors)   # avoid log(0)
    ax.semilogy(errors, 'o-', color='steelblue', ms=4, lw=1.5,
                label='|z_n - z*|')

    # Mark where DEQ "stops" (only needs z*, not the whole trajectory)
    ax.axvline(n_conv, color='darkorange', lw=1.5, ls='--',
               label=f'DEQ stops at n={n_conv}')
    ax.set_xlabel('Depth n  (= number of layers in explicit network)', fontsize=12)
    ax.set_ylabel('|z_n  -  z*|  (log scale)', fontsize=12)
    ax.set_title(
        f'Convergence to fixed point z*={z_star:.5f}\n'
        f'Spectral radius |f\' (z*)| = {abs(fp_deriv):.4f}  '
        f'  (< 1  =>  tame / NIP)',
        fontsize=11,
    )
    ax.legend(fontsize=10)
    ax.grid(True, alpha=0.25)

    # Right: explicit depth vs. DEQ output + gradient comparison
    ax2 = axes[1]
    depths = np.arange(len(orb))
    ax2.plot(depths, orb, 'o-', color='steelblue', ms=4, lw=1.5,
             label='Explicit z_N  (N layers)')
    ax2.axhline(z_star, color='limegreen', ls='--', lw=2,
                label=f'DEQ z* = {z_star:.5f}')

    textstr = (
        f'Implicit gradient  dz*/dw = sech²(wz*+x)·z* / (1 - f’(z*)):\n'
        f'  Implicit diff.: {grad_implicit:.6f}\n'
        f'  Finite diff.  : {grad_fd:.6f}\n'
        f'  Agreement: {abs(grad_implicit - grad_fd) < 1e-3}'
    )
    ax2.text(0.03, 0.05, textstr, transform=ax2.transAxes, fontsize=10,
             verticalalignment='bottom',
             bbox=dict(boxstyle='round', fc='lightyellow', alpha=0.9))

    ax2.set_xlabel('Number of explicit layers N', fontsize=12)
    ax2.set_ylabel('Output value', fontsize=12)
    ax2.set_title(
        'Explicit N-layer network vs. DEQ\n'
        'Both converge to z*; DEQ needs O(1) memory for backprop',
        fontsize=11,
    )
    ax2.legend(fontsize=10)
    ax2.grid(True, alpha=0.25)

    plt.tight_layout()
    plt.show()


## Connection to deep computations: existence, tameness, and NIP

### Existence of deep equilibria

The research paper (Dueñez et al., *Approx. Deep Computations*)
proves existence of deep equilibria for any **confined** computation —
one that stays in a compact region.  The key tool is the **Ellis–Numakura Lemma**:
every compact semigroup contains an idempotent element
(an element $e$ with $e \cdot e = e$).
Applied here: the space of “states reachable at infinite depth” forms a compact
semigroup, and its idempotents are exactly the deep equilibria.

This is a stronger statement than Banach: it guarantees *existence* without
requiring a contraction condition.  It does *not* guarantee uniqueness or stability.

### Stability = tameness = NIP

The **BFT dichotomy** (*Bourgain–Fremlin–Talagrand*) says:

> *A deep computation has a well-behaved limit (Baire class 1) if and only if
> it satisfies the NIP (No Independence Property).*

In the DEQ context:
| Regime | $\rho(J_f(z^\star))$ | Baire class | NIP | Behaviour |
|--------|:---:|:-----------:|:---:|-----------|
| Stable fixed point | $< 1$ | Baire-1 | ✓ | Convergent, unique, PAC-learnable |
| Marginal | $= 1$ | Baire-1 or higher | — | Slow / borderline |
| Unstable fixed point | $> 1$ | Not Baire-1 | ✗ | Oscillating or chaotic |

The phase diagram from the cobweb section is a **visual proof** of this theorem
in the 1-D case: the boundary between the green (stable) and red (unstable/oscillating)
regions is precisely where $|f'(z^\star)| = 1$.

### PAC-learnability

The NIP condition is equivalent (in the finitary sense) to **finite VC dimension**,
the standard condition guaranteeing PAC learnability via uniform convergence.
*Tame DEQ networks are PAC-learnable; wild ones need not be.*
